# Dynamic-Curvature Hyperbolic Continual Learning: GPU benchmark

This notebook compares **four methods** on two class-incremental benchmarks, with **real data only** and **no replay**:

| Method | Description |
|---|---|
| **Euclidean** | Euclidean prototype classifier, no protection against forgetting (lower bound) |
| **Euclidean + EWC** | Euclidean classifier + Elastic Weight Consolidation (Kirkpatrick et al., 2017) |
| **Hyperbolic (fixed c)** | HMCL-style: Lorentz head with fixed curvature + gradient projection + head pullback |
| **Dynamic curvature (ours)** | Learnable-curvature hyperbolic prototypes + Fisher protection + prototype drift compensation + old prototypes as negatives |

**Benchmarks:** Split-CIFAR100 and Split-MiniImageNet, each 10 tasks × 10 classes, downloaded from Kaggle.
**Metrics:** final accuracy, forgetting and backward transfer, under task-incremental (task id given) and class-incremental (no task id) evaluation.

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Get a Kaggle API token: kaggle.com → Settings → API → *Create New Token*. In Colab's left sidebar open **🔑 Secrets**, add `KAGGLE_USERNAME` and `KAGGLE_KEY`, and switch on *Notebook access*.
3. Run the steps **in order**. Results are saved to Google Drive after every run; if Colab disconnects, run Steps 1–3 again and the training steps continue where they stopped.

Measured on a T4, per seed, for all 4 methods: roughly 15–20 min for Split-CIFAR100 (10 epochs) and 29 min for Split-MiniImageNet (4 epochs). The first run also downloads MiniImageNet (6.3 GB) and resizes it once.


## Step 1: Setup
Mounts Google Drive (to store the datasets cache and the results), downloads the code from GitHub and reads the Kaggle key.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
DATA = f'{DRIVE}/hmcl_data'
RESULTS = f'{DRIVE}/hmcl_results_gpu'
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

REPO, BRANCH = 'https://github.com/Tarungowdap/HUBERILT-HMCL_ICML.git', 'main'
!rm -rf /content/hmcl && git clone -q --depth 1 -b {BRANCH} {REPO} /content/hmcl
%cd /content/hmcl
!pip install -q kagglehub
sys.path.insert(0, '/content/hmcl')

from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## Step 2: Load the datasets
Downloads **CIFAR-100** (`fedesoriano/cifar100`) and **MiniImageNet** (`arjunashok33/miniimagenet`) from Kaggle and splits each into **10 tasks of 10 classes**.
The first run resizes MiniImageNet to 64×64 and caches it in Drive; later runs load it in seconds.
Check the printout: each dataset should show **100 classes, 5000 train / 1000 test images per task**.

In [ ]:
import importlib, hmcl_cl as H
importlib.reload(H)
from pathlib import Path

DEVICE = torch.device('cuda')
BENCHMARKS = {
    'split_cifar100':     (10, 10, 32, 'fedesoriano/cifar100'),
    'split_miniimagenet': (10, 10, 64, 'arjunashok33/miniimagenet'),
}
NORM = {'split_cifar100': ((0.507, 0.487, 0.441), (0.267, 0.256, 0.276)),
        'split_miniimagenet': ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))}

def load_stream(ds, seed):
    tasks, cpt, size, slug = BENCHMARKS[ds]
    return H.build_stream(ds, Path(DATA), tasks, cpt, size, seed, DEVICE, 'kaggle', slug)

streams = {}
for ds in BENCHMARKS:
    print(f'\n--- {ds} ---')
    streams[ds] = load_stream(ds, seed=0)
print('\nDatasets ready.')


## Step 3: Experiment settings
The same settings are used for every method, so the comparison is fair. Only the method changes.
- `SEEDS`: use **3 seeds** for mean ± std and the significance test. Use `[0]` for a quicker first look.
- `EPOCHS`: per dataset, identical for all methods.

In [ ]:
import os, json, time

METHODS = ['euclidean', 'ewc', 'hmcl', 'dynamic-curvature']
SEEDS   = [0, 1]
EPOCHS  = {'split_cifar100': 10, 'split_miniimagenet': 4}
SETTINGS = dict(width=64, embed_dim=64, batch_size=128, lr=1e-3, amp=1)

def result_file(ds, m, seed):
    os.makedirs(f'{RESULTS}/{ds}', exist_ok=True)
    return f'{RESULTS}/{ds}/{m}__seed{seed}.json'

def train(ds, methods=METHODS, seeds=SEEDS):
    """Train every (method, seed) on one dataset. Finished runs are skipped, so this can be re-run safely."""
    t0 = time.time()
    for seed in seeds:
        stream = streams[ds] if seed == 0 else load_stream(ds, seed)
        for m in methods:
            f = result_file(ds, m, seed)
            if os.path.exists(f):
                print(f'skip {ds} / {m} / seed {seed} (done)'); continue
            aug = H.GPUAugment(*NORM[ds], pad=BENCHMARKS[ds][2] // 8)
            args = H.make_args(epochs=EPOCHS[ds], **SETTINGS)
            r = H.run_experiment(m, stream, args, seed, DEVICE, aug)
            r.update(dataset=ds, settings={**SETTINGS, 'epochs': EPOCHS[ds]})
            json.dump(r, open(f, 'w'), indent=1)
            print(f'   saved {f}  [{(time.time() - t0) / 60:.1f} min so far]')
        if seed != 0:
            del stream; torch.cuda.empty_cache()
    print(f'\n{ds}: done in {(time.time() - t0) / 60:.1f} min')

print('Methods:', [H.METHOD_LABELS[m] for m in METHODS], '| seeds:', SEEDS, '| epochs:', EPOCHS)


## Step 4: Train on Split-CIFAR100
All 4 methods × all seeds.

In [ ]:
train('split_cifar100')


## Step 5: Train on Split-MiniImageNet
All 4 methods × all seeds.

In [ ]:
train('split_miniimagenet')


## Step 6: Results
For each dataset, from the real runs saved in Step 4 and Step 5:
1. **Results table:** mean ± std over seeds for every method, with a check that all methods used the same epochs and seeds.
2. **How much better is ours:** % less forgetting, accuracy gain and paired t-test p-value, compared with every baseline. A one-line verdict for the success metric (vs Euclidean + EWC).
3. **Charts:** forgetting reduction, forgetting and accuracy bars, accuracy after each task, and a histogram of each old task's forgetting.

Everything is also saved to `MyDrive/hmcl_results_gpu/report`.

In [ ]:
import sys, os, importlib
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, '/content/hmcl'); import make_report as MR; importlib.reload(MR)

REPORT = f'{RESULTS}/report'
OURS = MR.OURS
runs = MR.load(RESULTS)
assert runs, f'No results found in {RESULTS}/<dataset>/ - run the training cell first'

for ds in ['split_cifar100', 'split_miniimagenet']:
    if ds not in runs:
        print(f'{ds}: no results yet'); continue
    order = [m for m in MR.LABEL if m in runs[ds]]
    out = f'{REPORT}/{ds}'; os.makedirs(out, exist_ok=True)
    display(Markdown(f'# {MR.DS_NAME[ds]}'))

    epochs = {m: next(iter(runs[ds][m].values())).get('epochs') for m in order}
    seeds = {m: sorted(runs[ds][m]) for m in order}
    if len(set(epochs.values())) > 1: print('WARNING: methods trained with different epochs:', epochs)
    if len({tuple(s) for s in seeds.values()}) > 1: print('WARNING: methods have different seeds:', seeds)
    print(f'epochs: {set(epochs.values())} | seeds per method: {seeds}')

    t = MR.results_table(runs[ds], order); open(f'{out}/results_table.md', 'w').write(t)
    display(Markdown('### Results (mean ± std over seeds)\n' + t))

    refs = [m for m in order if m != OURS]
    if OURS in runs[ds] and refs:
        imps = {r: MR.improvement(runs[ds], r) for r in refs}
        it = MR.improvement_table(imps); open(f'{out}/improvement_table.md', 'w').write(it)
        display(Markdown('### How much better is Dynamic curvature (ours)?\n' + it))
        e = imps.get('ewc')
        if e:
            print('Success metric (vs Euclidean + EWC): '
                  + ' | '.join(f"{tag}: {e[p]['forg_reduction_pct']:.0f}% less forgetting"
                               + (f" (p={e[p]['p']:.4f})" if e[p]['p'] is not None else ' (p n/a: 1 seed)')
                               for p, tag in MR.PROTOS))
        MR.chart_improvement(imps, ds, f'{out}/improvement.png'); display(Image(f'{out}/improvement.png'))

    MR.chart_bars(runs[ds], order, ds, f'{out}/forgetting_accuracy.png')
    MR.chart_curves(runs[ds], order, ds, f'{out}/accuracy_over_tasks.png')
    MR.chart_task_forgetting(runs[ds], order, ds, f'{out}/forgetting_per_task.png')
    for png in ('forgetting_accuracy', 'accuracy_over_tasks', 'forgetting_per_task'):
        display(Image(f'{out}/{png}.png'))

print(f'\nTables and charts saved to {REPORT}')


## Step 7 (optional): Ablations of our method
Two extra runs per dataset and seed, showing *why* our method works:
- **Ours in Euclidean space:** same recipe with a Euclidean head. If this is worse, the hyperbolic geometry matters.
- **Ours with fixed curvature:** same recipe with c frozen. If this is worse, the *dynamic* curvature matters.

Run Step 6 again afterwards; the ablations then appear in the table and charts.

In [ ]:
ABLATIONS = ['ablation-euclidean', 'ablation-fixed-curvature']
for ds in BENCHMARKS:
    train(ds, methods=ABLATIONS)


## Step 8: Publish the results to GitHub
Writes every table and chart into the repo (`results/`) and into the **Results** section of `README.md`, then pushes them to a branch `gpu-results`.

Needs a GitHub token with write access to the repo: github.com → Settings → Developer settings → Personal access tokens. Add it to Colab **🔑 Secrets** as `GITHUB_TOKEN`, and set your name and email below. Afterwards, open the link printed at the end and merge the pull request.

In [ ]:
import subprocess
from google.colab import userdata
GIT_NAME, GIT_EMAIL = 'your-github-username', 'your-github-email'
TOKEN = userdata.get('GITHUB_TOKEN')

NOTE = (f"GPU run: Split-CIFAR100 ({EPOCHS['split_cifar100']} epochs) and Split-MiniImageNet "
        f"({EPOCHS['split_miniimagenet']} epochs), 10 tasks x 10 classes, all images per task, "
        f"encoder width {SETTINGS['width']}, seeds {SEEDS}. Real data, zero replay.")
%cd /content/hmcl
subprocess.run(['python', 'make_report.py', '--results', RESULTS, '--out', 'results/gpu',
                '--readme', 'README.md', '--note', NOTE], check=True)

!git config user.name "{GIT_NAME}" && git config user.email "{GIT_EMAIL}"
!git checkout -q -B gpu-results && git add results README.md && git commit -q -m "Add GPU benchmark results (tables, charts, README)"
!git push -q -f https://{TOKEN}@github.com/Tarungowdap/HUBERILT-HMCL_ICML.git gpu-results
print('Now open https://github.com/Tarungowdap/HUBERILT-HMCL_ICML/compare/main...gpu-results and create + merge the pull request.')
